[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dilru1/microlensing-dtw/blob/main/notebooks/01_look_at_the_data.ipynb)

# Step 1 — Look at the data

Before any machine learning: what do microlensing events and their look-alikes actually look like in a Roman-like survey?

We simulate a **10-day window** sampled every **30 minutes** (480 epochs), flux normalised to the star's baseline, with **1% Gaussian noise**. Five classes:

| Class | What it is | Key parameter |
|---|---|---|
| constant | a quiet star | — |
| variable | a periodic variable star | period 0.2–5 d |
| stellar event | microlensing by a star (point lens) | t_E = 3–60 d |
| FFP event | microlensing by a free-floating planet, with finite-source effects | t_E = 1.2–12 h, ρ = 0.1–3 |
| flare | stellar flare (fast rise, slow decay) | half-width 15 min–5 h |

The simulator is in `mldtw/simulate.py`. Read it first; it is short.

In [ ]:
# Setup: works both locally and in Google Colab
import sys, os, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tslearn", "VBMicrolensing"], check=True)
    if not os.path.exists("microlensing-dtw"):
        subprocess.run(["git", "clone", "-q", "https://github.com/Dilru1/microlensing-dtw.git"], check=True)
    sys.path.insert(0, "microlensing-dtw")
    FIG_DIR = "microlensing-dtw/figures"
else:
    sys.path.insert(0, "..")
    FIG_DIR = "../figures"

import numpy as np
import matplotlib.pyplot as plt
from mldtw.simulate import make_dataset, CLASSES, T, SIGMA
from mldtw import tools


## 1.1 One example per class

In [ ]:
X, y, params = make_dataset(1, seed=42)

fig, axes = plt.subplots(5, 1, figsize=(9, 10), sharex=True)
for k, ax in enumerate(axes):
    ax.plot(T, X[k], ".", ms=2, color="k")
    info = ", ".join(f"{n}={v:.2f}" for n, v in params[k].items())
    ax.set_title(f"{CLASSES[k]}   {info}", fontsize=8, loc="left")
    ax.set_ylabel("flux / baseline")
axes[-1].set_xlabel("time [days]")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/step1_examples.png", dpi=150); plt.show()

**Look for:** the stellar event is a slow, symmetric bump (often only part of it fits in 10 days); the FFP event lasts hours; the flare rises fast and decays slowly.

## 1.2 Zoom on the short events

Free-floating-planet events and flares both last hours. Zoom in to compare their shapes.

In [ ]:
Xs, ys, ps = make_dataset(4, seed=7, classes=[3, 4])

fig, axes = plt.subplots(2, 4, figsize=(12, 5), sharey="row")
for i in range(8):
    ax = axes.flat[i]
    t0 = ps[i]["t0"]
    m = np.abs(T - t0) < 0.6
    ax.plot((T[m] - t0) * 24, Xs[i][m], ".-", ms=3, lw=0.5)
    ax.set_title(CLASSES[ys[i]], fontsize=9)
    ax.set_xlabel("hours from peak")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/step1_short_events.png", dpi=150); plt.show()

**Look for:** FFP events are **symmetric** (flat-topped when ρ ≳ 1); flares are **asymmetric**. With only 30-minute sampling, a 1-hour event has only a few points, which is why short events are hard.

## 1.3 Duration is the physics: same shape, different t_E

The Einstein time t_E scales with √(lens mass). Here is one event shape with t_E multiplied by 10× and 100×.

In [ ]:
from mldtw.simulate import pspl_mag

fig, ax = plt.subplots(figsize=(8, 3.5))
for tE in [0.3, 3, 30]:
    u = np.sqrt(0.1**2 + ((T - 5) / tE) ** 2)
    ax.plot(T, pspl_mag(u), label=f"t_E = {tE} d")
ax.set_yscale("log"); ax.set_xlabel("time [days]"); ax.set_ylabel("magnification")
ax.legend(); plt.tight_layout(); plt.savefig(f"{FIG_DIR}/step1_tE.png", dpi=150); plt.show()

**Key idea for the next steps:** a similarity measure that *stretches* time (like DTW) can make these three curves look alike, and so may throw away t_E, the very quantity that tells us the lens mass. A shift in t₀, on the other hand, is just a nuisance.

## 1.4 Simple features per class

Peak height vs. duration above half-maximum — a classic first EDA plot.

In [ ]:
X, y, _ = make_dataset(150, seed=1)

peak = X.max(axis=1) - 1
half = 1 + peak / 2
width_h = (X > half[:, None]).sum(axis=1) * 0.5      # hours above half-maximum (30-min cadence)

fig, ax = plt.subplots(figsize=(7, 5))
for k in range(5):
    m = y == k
    ax.scatter(width_h[m], peak[m], s=10, label=CLASSES[k], alpha=0.7)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("time above half-maximum [hours]"); ax.set_ylabel("peak flux excess")
ax.legend(); plt.tight_layout(); plt.savefig(f"{FIG_DIR}/step1_features.png", dpi=150); plt.show()

**Questions to think about before Step 2**

1. Which classes overlap in this plot? Why?
2. The variable stars and constant stars sit where? Is "peak height" meaningful for them?
3. What information does this 2-feature summary throw away that the full light curve keeps (e.g. symmetry)?

**Try:** change `SIGMA` in `simulate.py` to 0.003 or 0.03 and rerun. How do the short events change?